# 15 — Practice: Module 3 — Cleaning & Transformation

25 interview-style questions covering missing data, duplicates, replace, regex,
string operations, and datetime handling.

In [ ]:
import pandas as pd
import numpy as np
from io import StringIO

np.random.seed(42)
n = 300

# Customer Orders dataset with realistic messiness
orders = pd.DataFrame({
    'order_id':    range(1001, 1001 + n),
    'customer_id': np.random.randint(1, 100, n),
    'product':     np.random.choice(['Laptop', 'Phone', 'Tablet', 'Headphones', 'Charger'], n),
    'category':    np.random.choice(['Electronics', 'ELECTRONICS', 'electronics', 'Accessories'], n),
    'amount':      np.where(np.random.random(n) > 0.07,
                            np.round(np.random.uniform(500, 80000, n), 2),
                            np.nan),
    'city':        np.random.choice(['Mumbai', 'Delhi', None, 'Bangalore', 'Pune'], n,
                                    p=[0.25, 0.25, 0.1, 0.25, 0.15]),
    'status':      np.random.choice(['Delivered', 'DELIVERED', 'returned', 'Pending', None], n,
                                    p=[0.4, 0.2, 0.15, 0.2, 0.05]),
    'order_date':  pd.date_range('2022-01-01', periods=n, freq='D'),
    'email':       [f'user{i}@mail.com' if np.random.random() > 0.05 else None for i in range(n)],
    'phone_raw':   [f'({np.random.randint(70,99)}) {np.random.randint(1000,9999)}-{np.random.randint(1000,9999)}'
                    for _ in range(n)]
})

# Introduce duplicate rows
duplicates = orders.sample(15, random_state=42)
orders = pd.concat([orders, duplicates], ignore_index=True)

print(f"Dataset shape: {orders.shape}")
print(orders.dtypes)

---
### Q1: Count missing values per column, show percentage

In [ ]:
missing = pd.DataFrame({
    'count': orders.isna().sum(),
    'pct':   (orders.isna().mean() * 100).round(1)
})
print(missing[missing['count'] > 0].sort_values('count', ascending=False))

---
### Q2: Drop rows with any NaN vs only when a critical column is NaN

In [ ]:
# Drop ALL rows with any NaN
strict = orders.dropna()
print(f"Drop any NaN: {len(strict)} rows")

# Drop only when amount (critical) is NaN
relaxed = orders.dropna(subset=['amount'])
print(f"Drop only if amount NaN: {len(relaxed)} rows")

---
### Q3: Fill missing amount with the median by product category

In [ ]:
orders_filled = orders.copy()
orders_filled['amount'] = (
    orders_filled
    .groupby('product')['amount']
    .transform(lambda x: x.fillna(x.median()))
)
# Any remaining NaN (product group had all NaN): fill with global median
orders_filled['amount'] = orders_filled['amount'].fillna(orders_filled['amount'].median())

print(f"Amount NaN before: {orders['amount'].isna().sum()}")
print(f"Amount NaN after:  {orders_filled['amount'].isna().sum()}")

---
### Q4: Forward-fill missing city within each customer_id group

In [ ]:
df = orders.copy().sort_values(['customer_id', 'order_date'])

# ffill within each customer — assume they remain in the same city
df['city'] = df.groupby('customer_id')['city'].ffill().bfill()
# bfill() catches any that were NaN before the first valid value

print(f"City NaN before: {orders['city'].isna().sum()}")
print(f"City NaN after:  {df['city'].isna().sum()}")

---
### Q5: Count and remove exact duplicate rows

In [ ]:
n_dups = orders.duplicated().sum()
print(f"Exact duplicates: {n_dups}")

clean = orders.drop_duplicates().reset_index(drop=True)
print(f"After drop_duplicates(): {len(clean)}")

---
### Q6: Keep only the latest order per customer (dedup by customer, keep last)

In [ ]:
latest_order = (
    orders
    .sort_values('order_date')
    .drop_duplicates(subset=['customer_id'], keep='last')
    .reset_index(drop=True)
)
print(f"Unique customers: {orders['customer_id'].nunique()}")
print(f"Latest orders:    {len(latest_order)}")
print(latest_order[['customer_id', 'order_date', 'amount']].head())

---
### Q7: Normalize the category column (Engineering, SALES, sales → title case)

In [ ]:
print("Before:", orders['category'].value_counts().to_dict())
orders['category'] = orders['category'].str.strip().str.title()
print("After: ", orders['category'].value_counts().to_dict())

---
### Q8: Normalize the status column using replace()

In [ ]:
status_map = {
    'DELIVERED': 'Delivered',
    'delivered': 'Delivered',
    'returned':  'Returned',
    'pending':   'Pending',
}
orders['status'] = orders['status'].replace(status_map).fillna('Unknown')
print(orders['status'].value_counts())

---
### Q9: Clean phone_raw column — extract digits only

In [ ]:
print("Raw sample:", orders['phone_raw'].head(3).tolist())

orders['phone_clean'] = orders['phone_raw'].str.replace(r'\D', '', regex=True)

print("Cleaned sample:", orders['phone_clean'].head(3).tolist())
print("Lengths:", orders['phone_clean'].str.len().value_counts().head())

---
### Q10: Extract domain from email column using regex

In [ ]:
orders['email_domain'] = orders['email'].str.extract(r'@([\w\.]+)', expand=False)
print(orders[['email', 'email_domain']].dropna().head(5))
print(orders['email_domain'].value_counts().head())

---
### Q11: Find all orders with an amount > 2 standard deviations from mean

In [ ]:
valid = orders.dropna(subset=['amount'])
mean, std = valid['amount'].mean(), valid['amount'].std()

outliers = valid[np.abs(valid['amount'] - mean) > 2 * std]
print(f"Outliers (>2σ): {len(outliers)}")
print(outliers[['order_id', 'product', 'amount']].head())

---
### Q12: Extract year, month, day_of_week from order_date

In [ ]:
orders['year']        = orders['order_date'].dt.year
orders['month']       = orders['order_date'].dt.month
orders['day_of_week'] = orders['order_date'].dt.day_name()

print(orders[['order_date', 'year', 'month', 'day_of_week']].head(5))

---
### Q13: Calculate days since order_date to a reference date

In [ ]:
reference = pd.Timestamp('2023-01-01')
orders['days_since_order'] = (reference - orders['order_date']).dt.days
print(orders[['order_id', 'order_date', 'days_since_order']].head(5))

---
### Q14: Resample orders by week — count and total revenue

In [ ]:
ts = orders.dropna(subset=['amount']).set_index('order_date').sort_index()

weekly = ts.resample('W').agg(
    orders=('order_id', 'count'),
    revenue=('amount', 'sum')
).round(0)
print(weekly.head(8))

---
### Q15: Find orders placed on weekends

In [ ]:
weekend_orders = orders[orders['order_date'].dt.day_of_week >= 5]
print(f"Weekend orders: {len(weekend_orders)} ({len(weekend_orders)/len(orders):.1%})")
print(weekend_orders['order_date'].dt.day_name().value_counts())

---
### Q16: str.contains to find orders with 'phone' or 'tablet' in product name (case insensitive)

In [ ]:
mobile_orders = orders[orders['product'].str.contains('Phone|Tablet', case=False, regex=True)]
print(f"Phone/Tablet orders: {len(mobile_orders)}")
print(mobile_orders['product'].value_counts())

---
### Q17: Use str.split to extract username from email

In [ ]:
orders['username'] = orders['email'].str.split('@', expand=True)[0]
print(orders[['email', 'username']].dropna().head(5))

---
### Q18: Validate emails with str.match regex

In [ ]:
email_pattern = r'^[\w.%+\-]+@[\w.\-]+\.[a-zA-Z]{2,}$'
orders['email_valid'] = orders['email'].str.match(email_pattern, na=False)

print(f"Valid emails:   {orders['email_valid'].sum()}")
print(f"Invalid/NaN:   {(~orders['email_valid']).sum()}")

---
### Q19: Use interpolate to fill gaps in a time series

In [ ]:
daily_rev = pd.Series(
    [10000, np.nan, np.nan, 13000, np.nan, 15000, 16000, np.nan, 18000],
    index=pd.date_range('2023-01-01', periods=9, freq='D')
)

result = pd.DataFrame({
    'original':    daily_rev,
    'interpolated': daily_rev.interpolate('linear')
})
print(result)

---
### Q20: Use dropna(thresh=) to keep only rows with at least 7 non-null values

In [ ]:
total_cols = len(orders.columns)
before = len(orders)

threshold = 7
clean = orders.dropna(thresh=threshold)

print(f"Total columns: {total_cols}")
print(f"Before: {before}  After (thresh={threshold}): {len(clean)}")

---
### Q21: Build a complete cleaning pipeline for the orders dataset

In [ ]:
def clean_orders(df: pd.DataFrame) -> pd.DataFrame:
    return (
        df
        .drop_duplicates()
        .assign(
            category   = lambda df: df['category'].str.strip().str.title(),
            status     = lambda df: df['status'].str.strip().str.title().fillna('Unknown'),
            city       = lambda df: df['city'].fillna('Unknown'),
            amount     = lambda df: df.groupby('product')['amount'].transform(
                             lambda x: x.fillna(x.median())
                         ).fillna(df['amount'].median()),
            phone_clean= lambda df: df['phone_raw'].str.replace(r'\D', '', regex=True),
            year       = lambda df: df['order_date'].dt.year,
            month      = lambda df: df['order_date'].dt.month,
            is_weekend = lambda df: df['order_date'].dt.day_of_week >= 5,
        )
        .reset_index(drop=True)
    )

cleaned = clean_orders(orders)
print(f"Cleaned shape: {cleaned.shape}")
print(f"Remaining NaN:\n{cleaned.isna().sum()[cleaned.isna().sum() > 0]}")

---
### Q22: Convert duration string ('2h 30m') to total minutes

In [ ]:
durations = pd.Series(['2h 30m', '1h 15m', '45m', '3h', '0h 5m'])

hours   = durations.str.extract(r'(\d+)h').fillna(0).astype(int)[0]
minutes = durations.str.extract(r'(\d+)m').fillna(0).astype(int)[0]

total_minutes = hours * 60 + minutes
result = pd.DataFrame({'duration': durations, 'total_minutes': total_minutes})
print(result)

---
### Q23: Demonstrate the difference between ffill() and bfill() on NaN

In [ ]:
s = pd.Series([np.nan, 2, np.nan, np.nan, 5, np.nan])
print(pd.DataFrame({
    'original': s,
    'ffill':    s.ffill(),
    'bfill':    s.bfill(),
    'mean_fill': s.fillna(s.mean())
}))

---
### Q24: Use str.get_dummies to one-hot encode a comma-separated column

In [ ]:
df = pd.DataFrame({
    'customer': ['Alice', 'Bob', 'Carol', 'Dave'],
    'tags':     ['premium,loyal', 'new', 'premium,discount', 'loyal,at-risk']
})

dummies = df['tags'].str.get_dummies(sep=',')
result = pd.concat([df, dummies], axis=1)
print(result)

---
### Q25: Parse period from order_date, aggregate revenue per quarter

In [ ]:
orders_valid = orders.dropna(subset=['amount']).copy()
orders_valid['quarter'] = orders_valid['order_date'].dt.to_period('Q')

quarterly = (
    orders_valid
    .groupby('quarter')['amount']
    .agg(count='count', total='sum', avg='mean')
    .round(0)
)
print(quarterly)

---
## Module 3 Summary

| Topic | Key Pattern |
|-------|-------------|
| Missing detection | `df.isna().sum()` + `df.isna().mean()` |
| Group-aware fill | `groupby().transform(lambda x: x.fillna(x.median()))` |
| Time series fill | `ffill()` → `bfill()` → scalar |
| Dedup | `sort_values().drop_duplicates(keep='last')` for latest record |
| String normalize | `.str.strip().str.title()` + `.replace(mapping_dict)` |
| Phone/number clean | `.str.replace(r'\D', '', regex=True)` |
| Regex extract | `.str.extract(r'(?P<name>pattern)')` |
| DateTime features | `.dt.year`, `.dt.month`, `.dt.day_of_week`, `.dt.quarter` |
| Resample | `.resample('W').agg(...)` on DatetimeIndex |
| Tags → one-hot | `.str.get_dummies(sep=',')` |